# Plank Analysis
**Metrics tracked:**
1. Body alignment angle (shoulder → hip → ankle) — detects sagging or piking
2. Hold duration timer
3. Hip sag / hip pike detection
4. Head position (neck alignment)
5. Elbow angle (shoulder → elbow → wrist) — works for side-view camera

**Alerts:**
- 🔴 Hip Sag — body angle drops below 160°
- 🔴 Hip Pike — body angle rises above 195°
- 🔴 Head Drop — neck angle below 155°
- 🔴 Elbow Angle Off — elbow not in 80°–100° range

**Note:** Elbow distance is still stored in telemetry data but the plot uses elbow angle instead, which works correctly from a side-view camera.

In [ ]:
!pip install mediapipe==0.10.20 opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 23.3 MB/s eta 0:00:00

In [ ]:
import mediapipe as mp
print(mp.__version__)
print('solutions' in dir(mp))
mp_pose = mp.solutions.pose
print('Pose Ready')

0.10.20
True
Pose Ready


In [ ]:
import kagglehub

path = kagglehub.dataset_download('hasyimabdillah/workoutfitness-video')
print('Path to dataset files:', path)

# List available plank videos
!ls {path}/plank

100%|██████████| 4.32G/4.32G [02:29<00:00, 31.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/hasyimabdillah/workoutfitness-video/versions/5
plank_1.MOV  plank_3.mp4  plank_5.mp4  plank_7.mp4
plank_2.mp4  plank_4.mp4  plank_6.mp4


## Live Preview

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
from google.colab.patches import cv2_imshow

mp_pose = mp.solutions.pose

# ── SETTINGS ──────────────────────────────────────────────────────────────
# Body alignment: shoulder-hip-ankle angle
# A perfect plank = ~180° (straight line)
ALIGNMENT_MIN   = 160   # below this → hip sag
ALIGNMENT_MAX   = 195   # above this → hip pike
HEAD_DROP_ANGLE = 155   # shoulder-ear angle below this → head drop
ELBOW_WIDE_DIST = 0.25  # elbow separation > 25% frame width → flare
GRAPH_HISTORY   = 120

# ── ANGLE FUNCTION ──────────────────────────────────────────────────────
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

# ── GRAPH OVERLAY ───────────────────────────────────────────────────────
def draw_graph(frame, history, color=(0,255,255)):
    gh, gw = 100, 280
    x0 = 20
    y0 = frame.shape[0] - gh - 20
    cv2.rectangle(frame, (x0, y0), (x0+gw, y0+gh), (20,20,20), -1)
    if len(history) > 1:
        for i in range(1, len(history)):
            x1 = x0 + int((i-1) * gw / GRAPH_HISTORY)
            y1 = y0 + gh - int(np.clip((history[i-1]-140)/60, 0, 1) * gh)
            x2 = x0 + int(i * gw / GRAPH_HISTORY)
            y2 = y0 + gh - int(np.clip((history[i]-140)/60, 0, 1) * gh)
            cv2.line(frame, (x1,y1), (x2,y2), color, 2)

# ── VIDEO ───────────────────────────────────────────────────────────────
cap  = cv2.VideoCapture(path + '/plank/plank_1.mp4')
pose = mp_pose.Pose()

hold_start     = None
hold_duration  = 0
angle_history  = deque(maxlen=GRAPH_HISTORY)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    rgb          = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results      = pose.process(rgb)

    if results.pose_landmarks:
        lm   = results.pose_landmarks.landmark
        h, w, _ = frame.shape

        def pt(id): return (int(lm[id].x * w), int(lm[id].y * h))
        def pt_arr(id): return np.array([lm[id].x * w, lm[id].y * h])

        # Key landmarks
        l_shoulder = pt_arr(11); r_shoulder = pt_arr(12)
        l_hip      = pt_arr(23); r_hip      = pt_arr(24)
        l_ankle    = pt_arr(27); r_ankle    = pt_arr(28)
        l_elbow    = pt_arr(13); r_elbow    = pt_arr(14)
        l_ear      = pt_arr(7);  r_ear      = pt_arr(8)

        # Midpoints
        shoulder_mid = (l_shoulder + r_shoulder) / 2
        hip_mid      = (l_hip + r_hip) / 2
        ankle_mid    = (l_ankle + r_ankle) / 2
        ear_mid      = (l_ear + r_ear) / 2

        # ── BODY ALIGNMENT ANGLE (shoulder-hip-ankle) ──
        body_angle = calculate_angle(shoulder_mid, hip_mid, ankle_mid)
        angle_history.append(body_angle)

        # ── HOLD TIMER ──
        if hold_start is None:
            hold_start = current_time
        hold_duration = current_time - hold_start

        # ── HEAD ANGLE (ear-shoulder alignment) ──
        head_angle = calculate_angle(ear_mid, shoulder_mid, hip_mid)

        # ── ELBOW WIDTH ──
        elbow_dist = abs(l_elbow[0] - r_elbow[0]) / w

        # ── DRAW SKELETON ──
        for p1, p2 in [
            (pt(11), pt(23)), (pt(23), pt(27)),   # left side
            (pt(12), pt(24)), (pt(24), pt(28)),   # right side
            (pt(11), pt(12)),                     # shoulders
            (pt(23), pt(24)),                     # hips
            (pt(11), pt(13)), (pt(12), pt(14)),   # upper arms
        ]:
            cv2.line(frame, p1, p2, (0,255,0), 2)

        # ── OVERLAYS ──
        cv2.putText(frame, f'Body Angle: {int(body_angle)}',
                    (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(frame, f'Hold: {hold_duration:.1f}s',
                    (30, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

        # ── ALERTS ──
        warning = ''
        if body_angle < ALIGNMENT_MIN:
            warning = 'Hip Sag!'
        elif body_angle > ALIGNMENT_MAX:
            warning = 'Hip Pike!'
        elif head_angle < HEAD_DROP_ANGLE:
            warning = 'Head Drop!'
        elif elbow_dist > ELBOW_WIDE_DIST:
            warning = 'Elbow Flare!'

        if warning:
            cv2.putText(frame, warning, (30, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

        draw_graph(frame, angle_history)

    cv2_imshow(frame)
    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()

## Save Processed Video

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

mp_pose = mp.solutions.pose

ALIGNMENT_MIN   = 160
ALIGNMENT_MAX   = 195
HEAD_DROP_ANGLE = 155
ELBOW_WIDE_DIST = 0.25

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

cap    = cv2.VideoCapture(path + '/plank/plank_4.mp4')
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter('plank_ANALYZED.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose = mp_pose.Pose()

hold_start = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    rgb          = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results      = pose.process(rgb)

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id):     return (int(lm[id].x*width), int(lm[id].y*height))
        def pt_arr(id): return np.array([lm[id].x*width, lm[id].y*height])

        l_shoulder = pt_arr(11); r_shoulder = pt_arr(12)
        l_hip      = pt_arr(23); r_hip      = pt_arr(24)
        l_ankle    = pt_arr(27); r_ankle    = pt_arr(28)
        l_elbow    = pt_arr(13); r_elbow    = pt_arr(14)
        l_ear      = pt_arr(7);  r_ear      = pt_arr(8)

        shoulder_mid = (l_shoulder + r_shoulder) / 2
        hip_mid      = (l_hip + r_hip) / 2
        ankle_mid    = (l_ankle + r_ankle) / 2
        ear_mid      = (l_ear + r_ear) / 2

        body_angle = calculate_angle(shoulder_mid, hip_mid, ankle_mid)
        head_angle = calculate_angle(ear_mid, shoulder_mid, hip_mid)
        elbow_dist = abs(l_elbow[0] - r_elbow[0]) / width

        if hold_start is None:
            hold_start = current_time
        hold_duration = current_time - hold_start

        # Skeleton
        for p1, p2 in [
            (pt(11), pt(23)), (pt(23), pt(27)),
            (pt(12), pt(24)), (pt(24), pt(28)),
            (pt(11), pt(12)), (pt(23), pt(24)),
            (pt(11), pt(13)), (pt(12), pt(14)),
        ]:
            cv2.line(frame, p1, p2, (0,255,0), 2)

        # Text overlays
        cv2.putText(frame, f'Body Angle: {int(body_angle)}', (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(frame, f'Hold: {hold_duration:.1f}s',    (30,85),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)
        cv2.putText(frame, f'Head Angle: {int(head_angle)}', (30,120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)

        # Alerts
        alert = ''
        if body_angle < ALIGNMENT_MIN:       alert = 'Hip Sag!'
        elif body_angle > ALIGNMENT_MAX:     alert = 'Hip Pike!'
        elif head_angle < HEAD_DROP_ANGLE:   alert = 'Head Drop!'
        elif elbow_dist > ELBOW_WIDE_DIST:   alert = 'Elbow Flare!'
        if alert:
            cv2.putText(frame, alert, (30,155), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print('✅ Video saved successfully.')

#from google.colab import files
#files.download('plank_ANALYZED.mp4')

✅ Video saved successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Exercise Telemetry (Single Chart)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

mp_pose = mp.solutions.pose

# ── SETTINGS ──────────────────────────────────────────────────────────────
ALIGNMENT_MIN   = 160
ALIGNMENT_MAX   = 195
HEAD_DROP_ANGLE = 155
ELBOW_WIDE_DIST = 0.25   # kept for video overlay alert
ELBOW_ANGLE_MIN = 70     # elbow angle too bent
ELBOW_ANGLE_MAX = 100    # elbow angle too straight

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

cap    = cv2.VideoCapture(path + '/plank/plank_4.mp4')
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter('plank_ANALYZED.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose       = mp_pose.Pose()
hold_start = None
telem_data = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time   = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    rgb            = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results        = pose.process(rgb)

    body_angle      = 0
    head_angle      = 0
    elbow_dist      = 0
    avg_elbow_angle = 0
    hold_duration   = 0
    current_alerts  = []

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id):     return (int(lm[id].x*width), int(lm[id].y*height))
        def pt_arr(id): return np.array([lm[id].x*width, lm[id].y*height])

        l_shoulder = pt_arr(11); r_shoulder = pt_arr(12)
        l_hip      = pt_arr(23); r_hip      = pt_arr(24)
        l_ankle    = pt_arr(27); r_ankle    = pt_arr(28)
        l_elbow    = pt_arr(13); r_elbow    = pt_arr(14)
        l_ear      = pt_arr(7);  r_ear      = pt_arr(8)
        l_wrist    = pt_arr(15); r_wrist    = pt_arr(16)   # NEW

        shoulder_mid = (l_shoulder + r_shoulder) / 2
        hip_mid      = (l_hip + r_hip) / 2
        ankle_mid    = (l_ankle + r_ankle) / 2
        ear_mid      = (l_ear + r_ear) / 2

        body_angle = calculate_angle(shoulder_mid, hip_mid, ankle_mid)
        head_angle = calculate_angle(ear_mid, shoulder_mid, hip_mid)

        # Elbow dist — kept for video overlay
        elbow_dist = abs(l_elbow[0] - r_elbow[0]) / width

        # Elbow angle — shoulder→elbow→wrist (works for side-view camera)
        l_elbow_angle   = calculate_angle(l_shoulder, l_elbow, l_wrist)
        r_elbow_angle   = calculate_angle(r_shoulder, r_elbow, r_wrist)
        avg_elbow_angle = (l_elbow_angle + r_elbow_angle) / 2

        if hold_start is None: hold_start = current_time
        hold_duration = current_time - hold_start

        # ── ALERTS ──
        if body_angle < ALIGNMENT_MIN:   current_alerts.append('Hip Sag!')
        if body_angle > ALIGNMENT_MAX:   current_alerts.append('Hip Pike!')
        if head_angle < HEAD_DROP_ANGLE: current_alerts.append('Head Drop!')
        if avg_elbow_angle < ELBOW_ANGLE_MIN or avg_elbow_angle > ELBOW_ANGLE_MAX:
            current_alerts.append('Elbow Angle Off!')

        # ── SKELETON & OVERLAYS ──
        for p1, p2 in [(pt(11),pt(23)),(pt(23),pt(27)),(pt(12),pt(24)),(pt(24),pt(28)),
                       (pt(11),pt(12)),(pt(23),pt(24)),(pt(11),pt(13)),(pt(12),pt(14))]:
            cv2.line(frame, p1, p2, (0,255,0), 2)
        cv2.putText(frame, f'Body Angle: {int(body_angle)}',   (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(frame, f'Hold: {hold_duration:.1f}s',      (30,85),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)
        cv2.putText(frame, f'Elbow Angle: {int(avg_elbow_angle)}', (30,120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)
        if current_alerts:
            cv2.putText(frame, current_alerts[0], (30,155), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

    telem_data.append({
        'Time (s)':     round(current_time, 2),
        'Body Angle':   round(body_angle, 2),
        'Head Angle':   round(head_angle, 2),
        'Elbow Dist':   round(elbow_dist, 3),       # kept in data
        'Elbow Angle':  round(avg_elbow_angle, 2),  # NEW — used in plot
        'Hold (s)':     round(hold_duration, 2),
        'Alerts':       ' | '.join(current_alerts) if current_alerts else None
    })
    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print('✅ Video saved.')

# ── TELEMETRY PLOT ──
df = pd.DataFrame(telem_data)
df = df[df['Body Angle'] > 50]

fig = px.line(df, x='Time (s)', y='Body Angle', title='Plank Form Telemetry')
fig.add_hline(y=ALIGNMENT_MIN, line_dash='dash', line_color='orange',
              annotation_text='Sag Threshold')
fig.add_hline(y=ALIGNMENT_MAX, line_dash='dash', line_color='orange',
              annotation_text='Pike Threshold')
fig.update_traces(hovertemplate='Time: %{x}s<br>Angle: %{y}°')
fig.update_yaxes(range=[150, 205])

# Split alerts by type so each chart only shows its own alerts
df_body_alerts  = df[df['Alerts'].str.contains('Hip|Head', na=False)]
df_elbow_alerts = df[df['Alerts'].str.contains('Elbow', na=False)]

# Chart 1 — only Hip Sag / Hip Pike / Head Drop alerts
fig.add_trace(go.Scatter(
    x=df_body_alerts['Time (s)'], y=df_body_alerts['Body Angle'],
    mode='markers', marker=dict(color='red', size=7, symbol='x'),
    name='Body Alert',
    hovertext=df_body_alerts['Alerts'], hoverinfo='text+x+y'
))
fig.show()

✅ Video saved.


## Multi-Chart Biomechanics Dashboard

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

mp_pose = mp.solutions.pose

# ── SETTINGS ──────────────────────────────────────────────────────────────
ALIGNMENT_MIN   = 160
ALIGNMENT_MAX   = 195
HEAD_DROP_ANGLE = 155
ELBOW_WIDE_DIST = 0.25   # kept for video overlay
ELBOW_ANGLE_MIN = 70    # elbow angle too bent
ELBOW_ANGLE_MAX = 100    # elbow angle too straight

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

cap    = cv2.VideoCapture(path + '/plank/plank_4.mp4')
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter('plank_ANALYZED.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose       = mp_pose.Pose()
hold_start = None
telem_data = []
current_time = 0.0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time   = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    rgb            = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results        = pose.process(rgb)

    body_angle      = 0
    head_angle      = 0
    elbow_dist      = 0
    avg_elbow_angle = 0
    hip_height      = 0
    hold_duration   = 0
    current_alerts  = []

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id):     return (int(lm[id].x*width), int(lm[id].y*height))
        def pt_arr(id): return np.array([lm[id].x*width, lm[id].y*height])

        l_shoulder = pt_arr(11); r_shoulder = pt_arr(12)
        l_hip      = pt_arr(23); r_hip      = pt_arr(24)
        l_ankle    = pt_arr(27); r_ankle    = pt_arr(28)
        l_elbow    = pt_arr(13); r_elbow    = pt_arr(14)
        l_ear      = pt_arr(7);  r_ear      = pt_arr(8)
        l_wrist    = pt_arr(15); r_wrist    = pt_arr(16)   # NEW

        shoulder_mid = (l_shoulder + r_shoulder) / 2
        hip_mid      = (l_hip + r_hip) / 2
        ankle_mid    = (l_ankle + r_ankle) / 2
        ear_mid      = (l_ear + r_ear) / 2

        body_angle = calculate_angle(shoulder_mid, hip_mid, ankle_mid)
        head_angle = calculate_angle(ear_mid, shoulder_mid, hip_mid)

        # Elbow dist — kept in data, used in video overlay
        elbow_dist = abs(l_elbow[0] - r_elbow[0]) / width

        # Elbow angle — shoulder→elbow→wrist (works from side-view camera)
        l_elbow_angle   = calculate_angle(l_shoulder, l_elbow, l_wrist)
        r_elbow_angle   = calculate_angle(r_shoulder, r_elbow, r_wrist)
        avg_elbow_angle = (l_elbow_angle + r_elbow_angle) / 2

        # Hip height relative to ankle (normalised)
        hip_height = (ankle_mid[1] - hip_mid[1]) / height

        if hold_start is None: hold_start = current_time
        hold_duration = current_time - hold_start

        # ── ALERTS (continuous — fire every frame) ──
        if body_angle < ALIGNMENT_MIN:   current_alerts.append('Hip Sag!')
        if body_angle > ALIGNMENT_MAX:   current_alerts.append('Hip Pike!')
        if head_angle < HEAD_DROP_ANGLE: current_alerts.append('Head Drop!')
        if avg_elbow_angle < ELBOW_ANGLE_MIN or avg_elbow_angle > ELBOW_ANGLE_MAX:
            current_alerts.append('Elbow Angle Off!')

        # ── SKELETON & OVERLAYS ──
        for p1, p2 in [(pt(11),pt(23)),(pt(23),pt(27)),(pt(12),pt(24)),(pt(24),pt(28)),
                       (pt(11),pt(12)),(pt(23),pt(24)),(pt(11),pt(13)),(pt(12),pt(14))]:
            cv2.line(frame, p1, p2, (0,255,0), 2)
        cv2.putText(frame, f'Body Angle: {int(body_angle)}',      (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(frame, f'Hold: {hold_duration:.1f}s',         (30,85),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)
        cv2.putText(frame, f'Head Angle: {int(head_angle)}',      (30,120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)
        cv2.putText(frame, f'Elbow Angle: {int(avg_elbow_angle)}',(30,155), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
        if current_alerts:
            cv2.putText(frame, current_alerts[0], (30,190), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

    telem_data.append({
        'Time (s)':    round(current_time, 2),
        'Body Angle':  round(body_angle, 2),
        'Head Angle':  round(head_angle, 2),
        'Elbow Dist':  round(elbow_dist, 3),       # kept in data
        'Elbow Angle': round(avg_elbow_angle, 2),  # used in plot
        'Hip Height':  round(hip_height, 4),
        'Hold (s)':    round(hold_duration, 2),
        'Alerts':      ' | '.join(current_alerts) if current_alerts else None
    })
    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print('✅ Video saved. Generating dashboard...')

# ── BUILD DATAFRAME ───────────────────────────────────────────────────────
df = pd.DataFrame(telem_data)
df = df[df['Body Angle'] > 50]

# ── DASHBOARD ─────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=(
        '1. Body Alignment Angle & Form Alerts  (ideal = 160°–195°)',
        '2. Hip Height Tracker  (flat line = good plank)',
        '3. Head / Neck Angle  (lower = head dropping)',
        '4. Elbow Angle  (ideal = 80°–100° for forearm plank)'
    ),
    row_heights=[0.35, 0.2, 0.25, 0.2]
)

# ── Plot 1: Body Alignment ──
fig.add_trace(go.Scatter(
    x=df['Time (s)'], y=df['Body Angle'],
    name='Body Angle', line=dict(color='royalblue', width=2)
), row=1, col=1)
fig.add_hrect(
    y0=ALIGNMENT_MIN, y1=ALIGNMENT_MAX,
    fillcolor='rgba(0,255,0,0.07)', layer='below', line_width=0,
    annotation_text='Good Zone', annotation_position='top right',
    row=1, col=1
)
fig.add_hline(y=ALIGNMENT_MIN, line_dash='dash', line_color='orange',
              annotation_text='Sag Limit', row=1, col=1)
fig.add_hline(y=ALIGNMENT_MAX, line_dash='dash', line_color='orange',
              annotation_text='Pike Limit', row=1, col=1)

# Split alerts by type so each chart only shows its own alerts
df_body_alerts  = df[df['Alerts'].str.contains('Hip|Head', na=False)]
df_elbow_alerts = df[df['Alerts'].str.contains('Elbow', na=False)]

# Chart 1 — only Hip Sag / Hip Pike / Head Drop alerts
fig.add_trace(go.Scatter(
    x=df_body_alerts['Time (s)'], y=df_body_alerts['Body Angle'],
    mode='markers', marker=dict(color='red', size=7, symbol='x'),
    name='Body Alert',
    hovertext=df_body_alerts['Alerts'], hoverinfo='text+x+y'
), row=1, col=1)

# ── Plot 2: Hip Height ──
fig.add_trace(go.Scatter(
    x=df['Time (s)'], y=df['Hip Height'],
    name='Hip Height', line=dict(color='darkorange', width=2)
), row=2, col=1)
fig.add_hline(y=0, line_dash='dot', line_color='white',
              annotation_text='Level', row=2, col=1)

# ── Plot 3: Head Angle ──
fig.add_trace(go.Scatter(
    x=df['Time (s)'], y=df['Head Angle'],
    name='Head Angle', line=dict(color='mediumpurple', width=2)
), row=3, col=1)
fig.add_hline(y=HEAD_DROP_ANGLE, line_dash='dash', line_color='red',
              annotation_text='Drop Threshold', row=3, col=1)

# ── Plot 4: Elbow Angle (replaces elbow distance) ──
fig.add_trace(go.Scatter(
    x=df['Time (s)'], y=df['Elbow Angle'],
    name='Elbow Angle', line=dict(color='teal', width=2)
), row=4, col=1)

# Chart 4 — only Elbow Angle Off alerts
fig.add_trace(go.Scatter(
    x=df_elbow_alerts['Time (s)'], y=df_elbow_alerts['Elbow Angle'],
    mode='markers', marker=dict(color='red', size=7, symbol='x'),
    name='Elbow Alert',
    hovertext=df_elbow_alerts['Alerts'], hoverinfo='text+x+y'
), row=4, col=1)

fig.add_hline(y=ELBOW_ANGLE_MIN, line_dash='dash', line_color='orange',
              annotation_text='Too Bent', row=4, col=1)
fig.add_hline(y=ELBOW_ANGLE_MAX, line_dash='dash', line_color='orange',
              annotation_text='Too Straight', row=4, col=1)
# Good elbow zone shading
fig.add_hrect(
    y0=ELBOW_ANGLE_MIN, y1=ELBOW_ANGLE_MAX,
    fillcolor='rgba(0,255,0,0.07)', layer='below', line_width=0,
    annotation_text='Good Zone', annotation_position='top right',
    row=4, col=1
)

fig.update_layout(
    title_text='Plank Biomechanics & Performance Dashboard',
    height=1200, hovermode='x unified', showlegend=True, template='plotly_dark'
)
fig.update_yaxes(title_text='Degrees (°)', range=[150, 205], row=1, col=1)
fig.update_yaxes(title_text='Normalised Ht.',              row=2, col=1)
fig.update_yaxes(title_text='Degrees (°)',                 row=3, col=1)
fig.update_yaxes(title_text='Degrees (°)',                 row=4, col=1)
fig.update_xaxes(title_text='Time in Video (s)',           row=4, col=1)

fig.show()

✅ Video saved. Generating dashboard...
